In [ ]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from  pathlib import Path

In [116]:
### read all pdfs

def process_all_pdf(pdf_directory):

    all_documents=[]
    
    pdf_dir=Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files) } files to process "  )


    for pdf_file in pdf_files:
        print(f"processing {pdf_file.name}")
        
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents= loader.load()


            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']= 'pdf'

            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")    


        except Exception as e :
            print(f" Error: {e}")



    print(f"Total documents loaded {len(all_documents)}")        
    return all_documents


all_pdf_doc = process_all_pdf("../data")

found 2 files to process 
processing test2.pdf
loaded 32 pages
processing test1.pdf
loaded 27 pages
Total documents loaded 59


In [117]:
all_pdf_doc

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '2026-05-14T01:38:14+00:00', 'source': '../data/pdf/test2.pdf', 'file_path': '../data/pdf/test2.pdf', 'total_pages': 32, 'format': 'PDF 1.7', 'title': '"Like Taking the Path of Least Resistance": Exploring the Impact of LLM Interaction on the Creative Process of Programming', 'author': 'Zeinabsadat Saghi; Run Huang; Souti Chattopadhyay', 'subject': '-  Human-centered computing  ->  Empirical studies in HCI.HCI theory, concepts and models.', 'keywords': 'Creativity, problem solving, LLM-assisted programming, Human-AI collaboration', 'moddate': '2026-05-14T01:38:14+00:00', 'trapped': '', 'modDate': "D:20260514013814+00'00'", 'creationDate': "D:20260514013814+00'00'", 'page': 0, 'source_file': 'test2.pdf', 'file_type': 'pdf'}, page_content='"Like Taking the Path of Least Resistance": Exploring the Impact of LLM\nInteraction on the Creative Process of Programming\nZEINABSADAT SAGH

In [118]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n", " ",""]
    )

    split_docs=text_splitter.split_documents(documents)

    print(f"documents of length {len(documents)} into {len(split_docs)} chunks")

    if split_docs:
        print(f"content :{split_docs[0].page_content[:200]} ...")
        print(f"metadata :{split_docs[0].metadata}" )


    return split_docs    

In [119]:
chunks = split_documents(all_pdf_doc)

chunks

documents of length 59 into 246 chunks
content :"Like Taking the Path of Least Resistance": Exploring the Impact of LLM
Interaction on the Creative Process of Programming
ZEINABSADAT SAGHI, Thomas Lord Department of Computer Science, University of  ...
metadata :{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '2026-05-14T01:38:14+00:00', 'source': '../data/pdf/test2.pdf', 'file_path': '../data/pdf/test2.pdf', 'total_pages': 32, 'format': 'PDF 1.7', 'title': '"Like Taking the Path of Least Resistance": Exploring the Impact of LLM Interaction on the Creative Process of Programming', 'author': 'Zeinabsadat Saghi; Run Huang; Souti Chattopadhyay', 'subject': '-  Human-centered computing  ->  Empirical studies in HCI.HCI theory, concepts and models.', 'keywords': 'Creativity, problem solving, LLM-assisted programming, Human-AI collaboration', 'moddate': '2026-05-14T01:38:14+00:00', 'trapped': '', 'modDate': "D:20260514013814+00'00'", 'creationDate'

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '2026-05-14T01:38:14+00:00', 'source': '../data/pdf/test2.pdf', 'file_path': '../data/pdf/test2.pdf', 'total_pages': 32, 'format': 'PDF 1.7', 'title': '"Like Taking the Path of Least Resistance": Exploring the Impact of LLM Interaction on the Creative Process of Programming', 'author': 'Zeinabsadat Saghi; Run Huang; Souti Chattopadhyay', 'subject': '-  Human-centered computing  ->  Empirical studies in HCI.HCI theory, concepts and models.', 'keywords': 'Creativity, problem solving, LLM-assisted programming, Human-AI collaboration', 'moddate': '2026-05-14T01:38:14+00:00', 'trapped': '', 'modDate': "D:20260514013814+00'00'", 'creationDate': "D:20260514013814+00'00'", 'page': 0, 'source_file': 'test2.pdf', 'file_type': 'pdf'}, page_content='"Like Taking the Path of Least Resistance": Exploring the Impact of LLM\nInteraction on the Creative Process of Programming\nZEINABSADAT SAGH

In [120]:
import numpy as np  
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb import Settings
import uuid
from typing import List,Tuple,Dict,Any
from sklearn.metrics.pairwise import  cosine_similarity

In [121]:
class EmbeddingManager:

    def __init__(self,model_name="all-MiniLM-L6-v2"):

        """ Initialise mebdding manager"""


        self.model_name=model_name
        self.model=None
        self._load_model()


    def _load_model(self):
        """load sentence transformer model"""    
        try:

            print(f"loading embedding model {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"model loaded succesfully . embedding dimesion {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"error loading model :{self.model_name} : {(e)}")    
            raise

    def generate_embeddings(self , texts : List[str])->np.ndarray:

        """ genrate embedidngs fir list of texts """  

        if not self.model:
            raise ValueError("model not loaded")

        print(f"genrrate embeddings for {len(texts)} texts:")    

        embeddings= self.model.encode(texts,show_progress_bar=True)
        print(f"generate embeddings with shape : {embeddings.shape}")
        return embeddings




embedding_mng= EmbeddingManager()
embedding_mng    



loading embedding model all-MiniLM-L6-v2


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6333.86it/s]


model loaded succesfully . embedding dimesion 384


/tmp/ipykernel_30967/3714868800.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"model loaded succesfully . embedding dimesion {self.model.get_sentence_embedding_dimension()}")


In [122]:
class VectorStore:

    def __init__(self,collection_name: str="pdf_documents", persist_directory: str='../data/vector_store'):
        """ Initialise vector store"""


        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialise_store()


    def _initialise_store(self):

        """ INitialsise chcroma db client and collection """

        try:

            #create client

            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            ## get or create collection

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "pdf embedings for RAG"}
            )

            print(f"vector store instialise {self.collection_name}")
            print(f"existing document in collection : {self.collection.count()}")

        except Exception as e:
            print(f"error initialising vector : {(e)}")    
            raise
    


    def add_documents(self,documents:List[any] , embeddings : np.ndarray) :

        """ add socuments and their embeddings on vectore"""

        if(len(documents) != len(embeddings)):

            raise ValueError("no of doc must match no of embddings")



        print(f"Adding {len(documents)} to vector store :")

        # prepare data fpr chromadb


        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]    

        for i ,(doc,embedding) in enumerate(zip(documents,embeddings)):

            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)


            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']= len(doc.page_content)
            metadatas.append(metadata)


            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())



        # add to collrction
        try:

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text

            )    

        except Exception as e:

            print(f"erro adding documents to vectordb :{e}")    


vector_store=VectorStore()

vector_store


vector store instialise pdf_documents
existing document in collection : 1012


In [123]:
texts= [doc.page_content for doc in chunks]

embeddings=embedding_mng.generate_embeddings(texts)

genrrate embeddings for 246 texts:


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:07<00:00,  1.05it/s]

generate embeddings with shape : (246, 384)


In [127]:
vector_store.add_documents(chunks,embeddings)

Adding 246 to vector store :


In [165]:
class RAGRetriever:

    """ handles query based retreival from vector db"""

    def __init__(self,vector_store:VectorStore,embedding_mng:EmbeddingManager):

        self.vector_store=vector_store
        self.embedding_mng=embedding_mng
    

    def retrieve(self,query:str  ,top_k: int=5 , score_threshhold: float=0.0)-> List[Dict[str,any]]:


        print(f"retrieving doc for query :{query}")
        print(f"TOPK K :{top_k}  Threshhold:{score_threshhold}")


        query_embedding = self.embedding_mng.generate_embeddings([query])[0]



        try:

            results= self.vector_store.collection.query(

                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            
            )


            retrieved_docs=[]
            


            if results['documents'] and results['documents'][0]:
                documents= results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):

                    similarity_score= 1-distance

                    if similarity_score > score_threshhold:

                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank': i+1
                        })    


                print(f"retrieved {len(retrieved_docs)} after filtering")

            else: 
                    print("no doc found")


            return retrieved_docs

        except Exception as e:

            print(f"value error in retreival {e}")    

            return[]


rag_retriever=RAGRetriever(vector_store,embedding_mng)










In [174]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")



llm = ChatGroq(groq_api_key=groq_api_key,model_name='llama-3.3-70b-versatile',temperature='0.1',max_tokens=1024)



def rag_simple(query,rag_retriever,llm,top_k=3):
    _

    results=rag_retriever.retrieve(query,top_k=top_k)
    context=["\n\n".join(doc['content'] for doc in results ) if results else ""]

    if not context:
        return "no relevant text under this"


    prompt=f""" use the following context to answer question precisely.
       context:
       {context}
       
       
       Question: {query}
       
       Answer:        """


    response = llm.invoke([prompt.format(context=context,query=query)])

    return response.content
    



In [175]:
answer=rag_simple("how llm affect programmers",rag_retriever,llm)
print(answer)

retrieving doc for query :how llm affect programmers
TOPK K :3  Threshhold:0.0
genrrate embeddings for 1 texts:


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 99.91it/s]

generate embeddings with shape : (1, 384)
retrieved 3 after filtering


LLMs affect programmers in the following ways:

1. **Verification and Refining**: Programmers use LLMs to verify and refine their own solutions, rather than relying on LLMs to solve problems for them.
2. **Debugging**: LLMs help programmers debug their code, especially when they have limited time or are unable to generate a working solution due to bugs.
3. **Test Case Generation**: LLMs assist programmers in generating test cases for their current solutions, which helps in verifying the code.
4. **Creative Freedom**: Some programmers feel that using LLMs to solve problems can constrain their creativity, so they prefer to use LLMs only to build and refine their own solutions.

Overall, LLMs are used as a tool to support and augment the programming process, rather than replacing the programmer's own problem-solving abilities.


In [190]:

def rag_advance(query,rag_retriever,llm,top_k=5,min_score=0.2,return_context=False):
    _

    results=rag_retriever.retrieve(query,top_k=top_k,score_threshhold=min_score)



    if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
    else:
        context = "\n\n".join([doc['content'] for doc in results])
        sources = [{
            'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
            'page': doc['metadata'].get('page', 'unknown'),
            'score': doc['similarity_score'],
            'preview': doc['content'][:120] + '...'
            } for doc in results]

        confidence = max([doc['similarity_score'] for doc in results])    
   

    prompt=f""" use the following context to answer question precisely.
       context:
       {context}
       
       
       Question: {query}
       
       answer:        """


    response = llm.invoke([prompt.format(context=context,query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    
    if return_context:
        output['context'] = context
    return output
    



In [191]:
# Example usage:
result = rag_advance("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

retrieving doc for query :Hard Negative Mining Technqiues
TOPK K :3  Threshhold:0.1
genrrate embeddings for 1 texts:


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 69.80it/s]

generate embeddings with shape : (1, 384)
retrieved 3 after filtering


Answer: Hard Negative Mining Techniques are essential components in contrastive learning for retrieval model training. They include:

1. Asynchronous ANN indexing mechanism (ANCE): periodically updates hard negatives using checkpoint states.
2. Dynamic hard negative sampling strategy (Conan-Embedding and its v2 version): excludes and refreshes samples when their scores fall below a threshold.
3. Positive-aware negative mining (NV-Retriever): introduces TopK-MarginPos and TopKPercPos filtering criteria to minimize false negatives.
4. Adaptive margin-based mining strategies (LGAI-Embedding): employs ANNA IR as a teacher retrieval model to identify high-quality hard negatives and uses TopKPercPos filtering to eliminate false negatives.
Sources: [{'source': 'test1.pdf', 'page': 4, 'score': 0.10690808296203613, 'preview': 'QZhou-Embedding Technical Report\nKingsoft AI\n2.4\nHard Negative Mining Techniques\nHard negatives serve as essential compo...'}, {'source': 'test1.pdf', 'page': 4, 'sco

In [194]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshhold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Hard Negative Mining Technqiues", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

retrieving doc for query :Hard Negative Mining Technqiues
TOPK K :3  Threshhold:0.1
genrrate embeddings for 1 texts:


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 106.53it/s]

generate embeddings with shape : (1, 384)
retrieved 3 after filtering
Streaming answer:
Use the following context to answer the question concisely.
Context:
QZhou-Embedding Technical Report
Kingsoft AI
2.4
Hard Negative Mining Techniques
Hard negatives serve as essential components in contrastive learning for retrieval model
training. Early work like ANCE[46] proposed an asynchronous ANN indexing mech-
an

ism that periodically updates hard negatives using checkpoint states to maintain
optimally challenging samples. Conan-Embedding[24] and its v2 version implemented
a dynamic hard negative sampling strategy by excluding and refreshing samples when
their scores fall below a threshold. NV-Retriever[47] proposed positive-aware negative
mining, introducing TopK-MarginPos and TopKPercPos ﬁltering criteria to minimize
false negatives. LGAI-Embedding[17] built upon NV-Retriever’s strategy with adap-
tive margin-based mining strategies, employing ANNA IR as a teacher retrieval model
to identify high-quality hard negatives while using TopKPercPos ﬁltering to eliminate
false negatives.
3
Uniﬁed Multi-task Learning Framework

QZhou-Embedding Technical Report
Kingsoft AI
2.4
Hard Negative Mining Techniques
Hard negatives serve as essential components in contrastive learning for retrieval model
training. Early work like ANCE[46] proposed an asynchronous ANN indexing mech-
anism that periodically upda